In [1]:
%pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74/74 [crewai]ai] [crewai_tools]ocol]tform]ager]  WARNING: The scripts opentelemetry-bootstrap and opentelemetry-instrument are installed in '/voc/work/.local/bin' which is not on PATH.  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.dy satisfied: appdirs<2.0.0,>=1.4.4 in /usr/local/lib/python3.10/site-packages (from crewai==0.28.8) (1.4.4)Requirement already satisfied: click<9.0.0,>=8.1.7 in /usr/local/lib/python3.10/site-packages (from crewai==0.28.8) (8.1.7)Collecting embedchain<0.2.0,>=0.1.98 (from crewai==0.28.8)  Downloading embedchain-0.1.128-py3-none-any.whl.metadata (9.2 kB)Collecting instructor<0.6.0,>=0.5.2 (from crewai==0.28.8)  Downloading instructor-0.5.2-py3-none-any.whl.metadata (10 kB)Collecting langchain<0.2.0,>=0.1.10 (from crewai==0.28.8)  Downloading langchain-0.1.20-py3-none-any.whl.metadata (13 kB)Requirement already satisfied: openai<2.0.0,>=1.13.3 in /usr/

In [2]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [3]:
from crewai import Agent, Task, Crew

2026-05-28 08:01:55,054 [embedchain] [INFO] Swapped std-lib sqlite3 with pysqlite3 for ChromaDb compatibility. Your original version was 3.31.1.


In [4]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

import warnings
warnings.filterwarnings('ignore')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [5]:

Incident_classifier = Agent(
    role="incident Identification Specialist",
    goal="Correctly interpret and classify customer incidents.",
    backstory="You are trained in identifying and categorizing customer incidents.",
    llm = llm,
    verbose=True
)

Operation_Sla_Reasoning = Agent(
    role="Operation SLA Reasoning Specialist",
    goal="Search the knowledge base to find the best match for the incident.",
    backstory="You are skilled at navigating knowledge bases to find relevant information for incident resolution.",
    llm=llm,
    verbose=True
)

Action_recommendation_agent = Agent(
    role="Action Recommendation Specialist",
    goal="Draft clear and actionable support replies based on the incident classification and SLA reasoning.",
    backstory="You excel at translating technical incident details into customer-friendly support responses.",
    llm=llm,
    verbose=True
)

escalation_priority_decision_agent = Agent(
    role="Escalation Priority Decision Specialist",
    goal="Determine whether an incident requires escalation based on its classification and SLA reasoning.",
    backstory="You evaluate incident details to make informed decisions about escalation needs based on company policies, customer impact, and urgency.",
    llm=llm,
    verbose=True
)


In [8]:
# ---------------- Tasks ----------------
identify_incident = Task(
    description="""Analyze the incident reports: "{incident_reports}".
Identify:
- Incident type (shipment delays, inventory mismatches, vendor issues, or process failures)
- Priority level (low, normal, high)
- customer impact (minor, moderate, severe)
Output as JSON with keys: Incident type, urgency, customer impact, summary.""",
    agent=Incident_classifier,
    expected_output="A JSON object describing classified Incident details."
)

search_knowledge_base = Task(
    description="""Based on the classification above and incident reports: "{incident_reports}",
search the knowledge base and retrieve the closest matching solution.
If no solution exists, respond with: 'NO MATCH FOUND'.""",
    agent=Operation_Sla_Reasoning,
    expected_output="Best possible KB solution or 'NO MATCH FOUND'."
)

Action_recommendation = Task(
    description="""Using the classification and retrieved KB content for: "{incident_reports}",
create a clear, empathetic, and actionable support response for the customer.
- Uses plain language
- Offers specific next steps or solutions
- Maintains a helpful and understanding tone""",
    agent=Action_recommendation_agent,
    expected_output="A customer-friendly support response draft."
)

escalation_evaluation = Task(
    description="""Evaluate whether the AI response is sufficient.
Consider:
- Confidence level
- Urgency from incident classification, type of incident, and customer impact
- Whether solution was 'NO MATCH FOUND'

Output either:
- `resolved` if handled successfully
- `requires_escalation` with explanation and escalation notes""",
    agent=escalation_priority_decision_agent,
    expected_output="Resolution status + justification."
)

In [10]:

# ---------------- Crew Setup ----------------
cs_crew = Crew(
    agents=[Incident_classifier, Operation_Sla_Reasoning, Action_recommendation_agent, escalation_priority_decision_agent],
    tasks=[identify_incident, search_knowledge_base, Action_recommendation, escalation_evaluation],
    verbose=True
)

In [11]:
# ---------------- Run ----------------
# incident_reports = input("\nEnter incident reports: ")
incident_reports= """Inventory mismatch reported for order #12345. Customer expected 3 items but only received 2. This is the second time this issue has occurred with this product line in the past month. Customer is frustrated and needs a resolution ASAP."""

result = cs_crew.kickoff(inputs={"incident_reports": incident_reports})

print("\n--- FINAL SYSTEM RESPONSE ---\n")
print(result)

 [DEBUG]: == Working Agent: incident Identification Specialist
 [INFO]: == Starting Task: Analyze the incident reports: "Inventory mismatch reported for order #12345. Customer expected 3 items but only received 2. This is the second time this issue has occurred with this product line in the past month. Customer is frustrated and needs a resolution ASAP.".
Identify:
- Incident type (shipment delays, inventory mismatches, vendor issues, or process failures)
- Priority level (low, normal, high)
- customer impact (minor, moderate, severe)
Output as JSON with keys: Incident type, urgency, customer impact, summary.


> Entering new CrewAgentExecutor chain...
I need to analyze the incident report regarding the inventory mismatch for order #12345. The report indicates that the customer expected 3 items but only received 2, and this issue has occurred previously with the same product line. The customer is frustrated and requires a resolution as soon as possible. 

To classify this incident, I w